# Minimal Stage 1 + Stage 2 demo (synthetic note)

This notebook runs the **LLMSmokeDetector** pipeline end-to-end on **one synthetic discharge note**:

1. **Stage 1**: unstructured generation (`stage1.run_stage1`)
2. **Stage 2**: constrained decoding to one label (`stage2.run_stage2`)

Notes:
- You can change `STAGE1_MODEL_KEY` and `STAGE2_MODEL_KEY` depending on your available hardware.
- Stage 2 requires the **outlines** package (see `requirements.txt` / your environment).

In [1]:
import sys
from pathlib import Path

# Assumes this notebook is in the repo root (same folder as stage1.py, stage2.py, etc.)
repo_root = Path().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd

from stage1 import run_stage1
from stage2 import run_stage2

In [2]:
# Synthetic discharge note (edit as needed)
synthetic_note = """DISCHARGE SUMMARY

HPI:
52M with COPD and HTN admitted for acute SOB. He reports smoking 1 pack/day for 30 years.
States he tried quitting last year but restarted. Interested in cessation.

Social History:
Tobacco: current every day smoker, ~1 PPD x 30 years.
Alcohol: occasional.
Drugs: denies.

Assessment/Plan:
1) COPD exacerbation: treated with steroids and bronchodilators.
2) Tobacco use disorder: counseled, started nicotine patch, provided quitline information.

Dispo:
Home. Follow up with PCP in 1 week.
"""

df = pd.DataFrame(
    [
        {
            "pat_enc_csn_id": "SYNTHETIC_001",
            "full_discharge_summary": synthetic_note,
        }
    ]
)

df

,pat_enc_csn_id,full_discharge_summary
0,SYNTHETIC_001,DISCHARGE SUMMARY\n\nHPI:\n52M with COPD and H...


In [3]:
# Run Stage 1 (unstructured)
# Recommended: use a small model key for local testing (e.g., "1B" or "3B") if you are resource-constrained.
STAGE1_MODEL_KEY = "20B"
TEMPERATURE_STAGE1 = 1.0
REASONING_EFFORT = "low"  # set to None if your model does not support it

df_stage1 = run_stage1(
    df=df,
    model_key=STAGE1_MODEL_KEY,
    temperature=TEMPERATURE_STAGE1,
    reasoning_effort=REASONING_EFFORT,
    text_col="full_discharge_summary",
    out_csv=None,  # set a path to save
    window_tokens=2048,
    max_new_tokens=200,
)

df_stage1[["pat_enc_csn_id", "stage1_raw_output", "stage1_time_seconds"]]

Loaded 20B model with MXFP4 quantization from Hugging Face hub.
Device: NVIDIA A100-SXM4-40GB (index: 0)
Total memory: 39.38 GB
Reserved memory: 3.77 GB
Allocated memory: 3.75 GB
Free memory within reserved: 0.02 GB

----------------------------------------

Device: NVIDIA A100-SXM4-40GB (index: 1)
Total memory: 39.38 GB
Reserved memory: 4.49 GB
Allocated memory: 4.45 GB
Free memory within reserved: 0.04 GB

----------------------------------------

Device: NVIDIA A100-SXM4-40GB (index: 2)
Total memory: 39.38 GB
Reserved memory: 4.67 GB
Allocated memory: 4.64 GB
Free memory within reserved: 0.03 GB

----------------------------------------



,pat_enc_csn_id,stage1_raw_output,stage1_time_seconds
0,SYNTHETIC_001,Smoker,7.090095


In [4]:
# Run Stage 2 (structured label via constrained decoding)
# Stage 2 requires `outlines` to be installed in your environment.
STAGE2_MODEL_KEY = "8B"  # can be different from Stage 1
TEMPERATURE_STAGE2 = 1.0

df_stage2 = run_stage2(
    df=df_stage1,
    stage1_col="stage1_raw_output",
    out_csv=None,       # set a path to save
    keep_stage1=True,   # keep the Stage 1 raw output column
    structured_model_key=STAGE2_MODEL_KEY,
    max_new_tokens=10,
    temperature=TEMPERATURE_STAGE2,
)

df_stage2[["pat_enc_csn_id", "stage2_label", "stage2_time_seconds", "stage1_time_seconds", "stage1_raw_output"]]

Device: NVIDIA A100-SXM4-40GB (index: 0)
Total memory: 39.38 GB
Reserved memory: 8.45 GB
Allocated memory: 8.39 GB
Free memory within reserved: 0.05 GB

----------------------------------------

Device: NVIDIA A100-SXM4-40GB (index: 1)
Total memory: 39.38 GB
Reserved memory: 9.79 GB
Allocated memory: 9.74 GB
Free memory within reserved: 0.05 GB

----------------------------------------

Device: NVIDIA A100-SXM4-40GB (index: 2)
Total memory: 39.38 GB
Reserved memory: 9.79 GB
Allocated memory: 9.69 GB
Free memory within reserved: 0.10 GB

----------------------------------------



,pat_enc_csn_id,stage2_label,stage2_time_seconds,stage1_time_seconds,stage1_raw_output
0,SYNTHETIC_001,Smoker,1.728193,7.090095,Smoker


In [5]:
# Convenience: show just the key outputs
display_cols = [
    "pat_enc_csn_id",
    "stage2_label",
    "stage2_time_seconds",
    "stage1_time_seconds",
]
df_stage2[display_cols]

,pat_enc_csn_id,stage2_label,stage2_time_seconds,stage1_time_seconds
0,SYNTHETIC_001,Smoker,1.728193,7.090095
